In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import glob
import os
import gc
from tqdm import tqdm

print("🔧 Environment Setup")
print(f"   PyTorch Version: {torch.__version__}")
print(f"   CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = torch.device('cuda')
else:
    print("   ⚠️ No GPU found!")
    device = torch.device('cpu')

print(f"\n✅ Using device: {device}")


🔧 Environment Setup
   PyTorch Version: 2.12.0+cu130
   CUDA Available: True
   GPU: NVIDIA GeForce RTX 3090
   GPU Memory: 25.30 GB

✅ Using device: cuda


In [2]:
DATA_PATH = "/workspace/data/*.pt"

sample_files = sorted(glob.glob(DATA_PATH))
print(f"📂 Found {len(sample_files)} files")
print(f"   First: {os.path.basename(sample_files[0])}")
print(f"   Last: {os.path.basename(sample_files[-1])}")

sample_data = torch.load(sample_files[0])
print(f"\n📊 Data Structure:")
print(f"   Shape: {sample_data.shape}")
print(f"   Features (INPUT_SIZE): {sample_data.shape[1]}")

INPUT_SIZE = sample_data.shape[1]
print(f"\n✅ INPUT_SIZE = {INPUT_SIZE}")


📂 Found 230 files
   First: tensor_OFI_Enhanced_BTC_2023-05-16.pt
   Last: tensor_OFI_Enhanced_BTC_2023-12-31.pt

📊 Data Structure:
   Shape: torch.Size([527472, 7])
   Features (INPUT_SIZE): 7

✅ INPUT_SIZE = 7


In [3]:
class RegimeAdaptiveLSTM(nn.Module):
    """
    LSTM for Regime-Adaptive Market Making (RAMM)
    Predicts volatility and OFI to inform gamma in AS framework
    """
    
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(RegimeAdaptiveLSTM, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layers
        self.lstm = nn.LSTM(
            input_size, 
            hidden_size, 
            num_layers, 
            batch_first=True, 
            dropout=dropout
        )
        
        # Feature reconstruction head (auxiliary task)
        self.feature_head = nn.Linear(hidden_size, input_size)
        
        # Volatility prediction head (realized vol for next period)
        self.volatility_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Softplus()  # Ensures positive
        )
        
        # Order Flow Imbalance prediction head
        self.ofi_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()  # Range [-1, 1]
        )
    
    def forward(self, x):
        # x shape: (batch, seq_len, features)
        lstm_out, _ = self.lstm(x)
        
        # Use last timestep
        last_hidden = lstm_out[:, -1, :]
        
        # Predict features (reconstruction)
        features_pred = self.feature_head(last_hidden)
        
        # Predict volatility (market risk indicator)
        volatility_pred = self.volatility_head(last_hidden)
        
        # Predict OFI (toxic flow indicator)
        ofi_pred = self.ofi_head(last_hidden)
        
        return features_pred, volatility_pred, ofi_pred
    
    def calculate_gamma(self, volatility, ofi):
        """
        Calculate risk aversion parameter for AS framework
        NOT trained - derived from predictions
        
        Higher volatility → Higher gamma (defensive spreads)
        Higher |OFI| → Higher gamma (avoid toxic flow)
        """
        # Normalize and combine signals
        vol_component = volatility.squeeze()
        ofi_component = torch.abs(ofi.squeeze())
        
        # Map to gamma range [0.01, 5.0] (typical AS range)
        base_gamma = 0.1
        gamma = base_gamma + 3.0 * (0.6 * vol_component + 0.4 * ofi_component)
        
        # Clip to reasonable bounds
        gamma = torch.clamp(gamma, 0.01, 5.0)
        
        return gamma

print("✅ Regime-Adaptive LSTM architecture defined")


✅ Regime-Adaptive LSTM architecture defined


In [4]:
# ================================================================
#  DATA CONTRACT -- CHANGE 5
#
#  The .pt tensor files contain NORMALIZED, LOG-TRANSFORMED data.
#  Each tensor has shape [N, 7] with the following columns:
#
#    Column 0: Normalized log-price
#              Original: log(USD price), then z-scored
#              Range in data: approx -14 to +16
#
#    Column 1: Normalized volume / flow indicator
#              Centered around 0, small magnitude (~1e-5)
#
#    Column 2: Normalized order book imbalance
#              Wide range (-130 to +140)
#
#    Column 3: Normalized spread feature
#              Very small range (~+/-0.01)
#
#    Column 4: Normalized tick indicator
#              Very small range (~0 to 0.001)
#
#    Column 5: Normalized binary/event indicator
#              Values {0, 1}
#
#    Column 6: Normalized price change / OFI feature
#              Range approx -91 to +67
#
#  CRITICAL: The backtest MUST denormalize column 0 before using
#  prices for cash updates, execution, or MTM valuation.
#  See denormalization utilities in backtesting.ipynb.
#
#  Denormalization: real_price = exp(col0 * std + mean)
#  Parameters saved in normalization_params.json (see Change 6).
# ================================================================

class BTCTickDataset(Dataset):
    """
    Dataset for ALREADY NORMALIZED .pt tensor files.

    See DATA CONTRACT above for column definitions.
    The backtest must denormalize before execution and MTM.
    """
    
    def __init__(self, file_paths, seq_len=60, stride=150):
        self.seq_len = seq_len
        self.stride = stride
        self.sequences = []
        
        print(f"📂 Loading {len(file_paths)} files...")
        for file_path in tqdm(file_paths, desc="Loading data"):
            data = torch.load(file_path)
            
            # Data is ALREADY normalized - use as-is
            for i in range(0, len(data) - seq_len, stride):
                seq = data[i:i + seq_len]
                if len(seq) == seq_len:
                    self.sequences.append(seq)
        
        print(f"✅ Loaded {len(self.sequences):,} sequences")
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        
        # X = full sequence (already normalized)
        X = seq
        
        # y = last timestep features
        y_features = seq[-1]
        
        # === VOLATILITY: From normalized log-prices (Column 0) ===
        # Since prices are already log-normalized, use std of changes
        price_changes = seq[1:, 0] - seq[:-1, 0]
        y_volatility = torch.std(price_changes).clamp(1e-6, 10.0)
        
        # === OFI: From volume columns if available ===
        # Column 1 looks like volume/flow indicator (mean=0, centered)
        # Column 2 could be another volume feature
        
        if seq.shape[1] >= 3:
            # Use columns 1 and 2 as bid/ask indicators
            flow_indicator = seq[-10:, 1].mean()  # Recent flow
            imbalance = seq[-10:, 2].mean()  # Recent imbalance
            
            # Combine into OFI proxy
            y_ofi = torch.tanh((flow_indicator + imbalance * 0.1))
            y_ofi = torch.clamp(y_ofi, -0.95, 0.95)
        else:
            # Price momentum
            price_momentum = seq[-1, 0] - seq[0, 0]
            y_ofi = torch.tanh(price_momentum * 0.5)
        
        return X, y_features, y_volatility, y_ofi

print("✅ Dataset class defined (data already normalized)")


✅ Dataset class defined (data already normalized)


In [5]:
print("🏗️ Initializing Regime-Adaptive LSTM Model...")
print(f"   Input Size: {INPUT_SIZE}")
print(f"   Hidden Size: 128")
print(f"   Num Layers: 2")

class RegimeAdaptiveLSTM(nn.Module):
    """LSTM for Regime-Adaptive Market Making (for normalized data)"""
    
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(RegimeAdaptiveLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size, 
            hidden_size, 
            num_layers, 
            batch_first=True, 
            dropout=dropout
        )
        
        # Feature reconstruction (output range matches input: ~-20 to +100)
        self.feature_head = nn.Linear(hidden_size, input_size)
        
        # Volatility prediction (std of normalized price changes: 0-5 range)
        self.volatility_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Softplus()  # Positive only, will scale below
        )
        
        # OFI prediction (bounded -1 to 1)
        self.ofi_head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
            nn.Tanh()
        )
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]
        
        features_pred = self.feature_head(last_hidden)
        
        # Volatility: Softplus output * 5 = range [0, ~5]
        # Typical std of normalized price changes is 0.1-2.0
        volatility_pred = self.volatility_head(last_hidden)
        
        # OFI: Already bounded by tanh [-1, 1]
        ofi_pred = self.ofi_head(last_hidden)
        
        return features_pred, volatility_pred, ofi_pred
    
    def calculate_gamma(self, volatility, ofi):
        """
        Calculate gamma from predictions
        
        For normalized data:
        - Low volatility (< 0.5): gamma ~0.2-0.5
        - Medium volatility (0.5-1.5): gamma ~0.5-1.2
        - High volatility (> 1.5): gamma ~1.2-2.5
        """
        vol_normalized = torch.clamp(volatility.squeeze() / 2.0, 0, 1)
        ofi_normalized = torch.abs(ofi.squeeze())
        
        risk_score = 0.7 * vol_normalized + 0.3 * ofi_normalized
        
        # Map to gamma range [0.1, 2.5]
        gamma = 0.1 + 2.4 * torch.sigmoid(risk_score * 3)
        
        return gamma

model = RegimeAdaptiveLSTM(
    input_size=INPUT_SIZE,
    hidden_size=128,
    num_layers=2,
    dropout=0.2
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"   Total Parameters: {total_params:,}")

# Optimizer with moderate learning rate
optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-5)

criterion_features = nn.MSELoss()
criterion_volatility = nn.MSELoss()
criterion_ofi = nn.MSELoss()

print("\n✅ Model initialized!")
print("📊 Data is already normalized (log-transformed)")
print("🎯 Expected ranges:")
print("   - Volatility: 0.1 - 2.0")
print("   - OFI: -1.0 to 1.0")
print("   - Gamma: 0.1 - 2.5")


🏗️ Initializing Regime-Adaptive LSTM Model...
   Input Size: 7
   Hidden Size: 128
   Num Layers: 2
   Total Parameters: 219,785

✅ Model initialized!
📊 Data is already normalized (log-transformed)
🎯 Expected ranges:
   - Volatility: 0.1 - 2.0
   - OFI: -1.0 to 1.0
   - Gamma: 0.1 - 2.5


In [ ]:
all_files = sorted(glob.glob(DATA_PATH))

train_files = sorted(all_files)
val_files = []
print(f"Train files: {len(train_files)}")
print(f"Val files:   {len(val_files)}")

# ===== TRAINING PARAMETERS =====
CHUNK_SIZE = 60
MAX_EPOCHS = 50  # Allows model to continue until actual plateau
MIN_EPOCHS = 15  # Minimum epochs before early stopping kicks in
IMPROVEMENT_THRESHOLD = 0.01  # Stop when 3-epoch improvement < 1%

SEQ_LEN = 60
STRIDE = 150
BATCH_SIZE = 1024

history = {
    'train_loss': [],
    'volatility_mean': [],
    'ofi_mean': [],
    'gamma_mean': [],
}

best_loss = float('inf')

print(f"🚀 Training on {len(train_files)} files in chunks of {CHUNK_SIZE}")
print(f"   Total chunks per epoch: {len(train_files) // CHUNK_SIZE + 1}")
print(
    f"   Stopping condition: Improvement < {IMPROVEMENT_THRESHOLD * 100:.1f}% after Epoch {MIN_EPOCHS}\n"
)

for epoch in range(MAX_EPOCHS):
  print(f"{'='*60}")
  print(f"EPOCH {epoch+1}/{MAX_EPOCHS}")
  print(f"{'='*60}\n")

  epoch_loss = 0.0
  epoch_vol = 0.0
  epoch_ofi = 0.0
  epoch_gamma = 0.0
  total_batches = 0

  # Process files in chunks
  for chunk_idx in range(0, len(train_files), CHUNK_SIZE):
    chunk_files = train_files[chunk_idx : chunk_idx + CHUNK_SIZE]

    print(
        f"📂 Chunk {chunk_idx//CHUNK_SIZE + 1}/{len(train_files)//CHUNK_SIZE + 1}"
    )
    print(f"   Files: {chunk_idx + 1} to {chunk_idx + len(chunk_files)}")

    # Load chunk
    train_dataset = BTCTickDataset(chunk_files, seq_len=SEQ_LEN, stride=STRIDE)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )

    # Train on this chunk
    model.train()
    pbar = tqdm(train_loader, desc=f"  Training")

    for X_batch, y_features, y_volatility, y_ofi in pbar:
      X_batch = X_batch.to(device)
      y_features = y_features.to(device)
      y_volatility = y_volatility.to(device).unsqueeze(1)
      y_ofi = y_ofi.to(device).unsqueeze(1)

      optimizer.zero_grad()

      features_pred, volatility_pred, ofi_pred = model(X_batch)

      loss_features = criterion_features(features_pred, y_features)
      loss_volatility = criterion_volatility(volatility_pred, y_volatility)
      loss_ofi = criterion_ofi(ofi_pred, y_ofi)

      # Balanced multi-task loss
      loss = loss_features + 1.0 * loss_volatility + 0.5 * loss_ofi

      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
      optimizer.step()

      with torch.no_grad():
        gamma_batch = model.calculate_gamma(volatility_pred, ofi_pred)

      epoch_loss += loss.item()
      epoch_vol += volatility_pred.mean().item()
      epoch_ofi += ofi_pred.mean().item()
      epoch_gamma += gamma_batch.mean().item()
      total_batches += 1

      pbar.set_postfix({
          'loss': f'{loss.item():.4f}',
          'σ': f'{volatility_pred.mean().item():.3f}',
          'OFI': f'{ofi_pred.mean().item():.3f}',
          'γ': f'{gamma_batch.mean().item():.2f}',
      })

    del train_dataset, train_loader
    print(f"   ✅ Chunk {chunk_idx//CHUNK_SIZE + 1} complete\n")

  # === MEMORY CLEANUP ===
  gc.collect()
  torch.cuda.empty_cache()
  print(f"🧹 Memory cleared after epoch {epoch+1}\n")

  # Epoch metrics aggregation
  avg_loss = epoch_loss / total_batches
  avg_vol = epoch_vol / total_batches
  avg_ofi = epoch_ofi / total_batches
  avg_gamma = epoch_gamma / total_batches

  history['train_loss'].append(avg_loss)
  history['volatility_mean'].append(avg_vol)
  history['ofi_mean'].append(avg_ofi)
  history['gamma_mean'].append(avg_gamma)

  print(f"📊 Epoch {epoch+1} Complete:")
  print(f"   Loss: {avg_loss:.4f}")
  print(f"   Volatility (σ): {avg_vol:.3f}")
  print(f"   OFI: {avg_ofi:.3f}")
  print(f"   Gamma (γ): {avg_gamma:.2f} [derived]")

  # Checkpoint best model
  if avg_loss < best_loss:
    best_loss = avg_loss
    torch.save(model.state_dict(), '/workspace/best_model.pth')
    print(f"  ⭐ New best model saved (Loss: {best_loss:.4f})")

  if (epoch + 1) % 5 == 0:
    torch.save(model.state_dict(), f'/workspace/model_epoch_{epoch+1}.pth')
    print(f"  ✅ Checkpoint saved: model_epoch_{epoch+1}.pth")

  # Early stopping check based on relative window improvement
  if epoch >= MIN_EPOCHS - 1:
    recent_losses = history['train_loss'][-3:]
    if len(recent_losses) >= 3:
      improvement = (recent_losses[0] - recent_losses[-1]) / recent_losses[0]
      print(f"  Loss trend (last 3 epochs): {improvement*100:.2f}% improvement")

      if improvement < IMPROVEMENT_THRESHOLD:
        print(
            f"\n🛑 Loss improvement fell below {IMPROVEMENT_THRESHOLD*100:.1f}%. Stopping at epoch {epoch+1}."
        )
        break
      else:
        print(f"  🚀 Still improving (> 1%). Continuing...\n")

# Save final state
torch.save(model.state_dict(), '/workspace/final_model.pth')
print('=' * 60)
print('TRAINING COMPLETE!')
print(f"   Final Train Loss: {history['train_loss'][-1]:.4f}")
print(f"   Best Train Loss:  {best_loss:.4f}")
print(f"   Final Volatility: {history['volatility_mean'][-1]:.3f}")
print(f"   Final OFI:        {history['ofi_mean'][-1]:.3f}")
print(f"   Final Gamma:      {history['gamma_mean'][-1]:.2f}")
print(f"   Saved models: /workspace/best_model.pth & /workspace/final_model.pth")
print('=' * 60)

Train files: 230
Val files:   0
🚀 Training on 230 files in chunks of 60
   Total chunks per epoch: 4
   Stopping condition: Improvement < 1.0% after Epoch 15

EPOCH 1/50

📂 Chunk 1/4
   Files: 1 to 60
📂 Loading 60 files...


Loading data: 100%|██████████| 60/60 [00:03<00:00, 17.30it/s]


✅ Loaded 446,514 sequences


  Training: 100%|██████████| 437/437 [00:16<00:00, 26.49it/s, loss=0.0245, σ=0.001, OFI=0.400, γ=1.51] 


   ✅ Chunk 1 complete

📂 Chunk 2/4
   Files: 61 to 120
📂 Loading 60 files...


...

  Training: 100%|██████████| 288/288 [00:10<00:00, 26.31it/s, loss=0.0034, σ=0.002, OFI=0.480, γ=1.55]


   ✅ Chunk 2 complete

📂 Chunk 3/4
   Files: 121 to 180
📂 Loading 60 files...


Loading data: 100%|██████████| 60/60 [00:02<00:00, 23.85it/s]


✅ Loaded 431,689 sequences


  Training:  64%|██████▍   | 270/422 [00:10<00:05, 29.37it/s, loss=0.0040, σ=0.001, OFI=0.444, γ=1.53]